# RiskFlow trained on other ice-dammed lakes, tested at Shisper

Same network as RiskFlow. Three tests, each with three seeds whose probabilities are averaged (threshold 0.5):
1. other lakes only, applied to every Shisper scene;
2. main split: other lakes and Shisper up to 2021, tested on Shisper from 2022 onward;
3. leave one year out: other lakes and all Shisper years but one, tested on the held-out year.

Training uses class-balanced batches augmented on the fly (flips, rotations, crops and brightness), 100 batches of 32 scenes per epoch, early stopping with patience 6 and at most 30 epochs. Outbursts whose lake was not visible before drainage (less than 0.02 km² of water in the preceding 60 days) are left out, and 15 % of the other lakes are held out with the Shisper validation year to choose the number of epochs.

Criteria fixed before running: at least 5 of the 7 Shisper GLOF scenes caught with years held out, a false-alarm rate below 10 % and both 2022 scenes caught in the main split; otherwise the Shisper-only model is kept.

**Inputs:** `RiskFlow_multisite_scenes` (notebook 11) and `RiskFlow_Shisper/scenes_128.npz` (notebook 10). **Output:** `RiskFlow_multisite_results`. GPU runtime (T4), about 2 hours; finished runs are skipped on a rerun.

## Setup

In [ ]:
import os, json, time, glob, warnings
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore', category=UserWarning)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
MS_DIR = f'{DRIVE}/RiskFlow_multisite_scenes'
SH_DIR = f'{DRIVE}/RiskFlow_Shisper'
OUT = f'{DRIVE}/RiskFlow_multisite_results'
os.makedirs(f'{OUT}/runs', exist_ok=True)

SEEDS = [42, 1, 2]
MAX_EPOCHS, PATIENCE, BATCH, STEPS = 30, 6, 32, 100
IMG, WINDOW_DAYS, THRESH = 128, 30, 0.5
VAL_LAKE_FRAC, MIN_WATER_BEFORE = 0.15, 0.02
CRITERIA = dict(loyo_min_glof=5, max_false_alarm_rate=0.10, main_glof=2)
OUTBURSTS = ['2019-06-22', '2020-05-29', '2021-05-15', '2022-05-07']
print('GPU:', tf.config.list_physical_devices('GPU'))

## Data

In [ ]:
# Shisper: the cached scenes of notebook 10
C = np.load(f'{SH_DIR}/scenes_{IMG}.npz', allow_pickle=True)
XS = C['X'].astype('float32')
S = pd.DataFrame({'date': [str(d) for d in C['dates']]})
ob = [pd.Timestamp(o) for o in OUTBURSTS]
def since(d):
    v = [(pd.Timestamp(d) - o).days for o in ob if pd.Timestamp(d) >= o]
    return min(v) if v else np.nan
S['days_since_outburst'] = S.date.map(since)
S['label'] = (S.days_since_outburst <= WINDOW_DAYS).astype(int)
S['year'] = S.date.str[:4].astype(int)
S['fold'] = S.year.astype(str).where(S.year > 2019, '2018-19')
print(f'Shisper: {len(S)} scenes, {int(S.label.sum())} GLOF')

# other lakes
metas, Xs = [], []
for f in sorted(glob.glob(f'{MS_DIR}/*.npz')):
    z = np.load(f, allow_pickle=True)
    if len(z['date']) == 0:
        continue
    metas.append(pd.DataFrame(dict(lake_id=os.path.basename(f)[:-4], date=[str(d) for d in z['date']],
                                   label=z['label'].astype(int), event_id=[str(e) for e in z['event_id']],
                                   water_km2=z['water_km2'])))
    Xs.append(z['X'])
M = pd.concat(metas, ignore_index=True)
XO = np.concatenate(Xs).astype('float32'); del Xs
M['region'] = M.lake_id.str[6:8]
print(f'other lakes: {M.lake_id.nunique()} lakes, {len(M)} scenes, {int(M.label.sum())} GLOF from {M[M.label == 1].event_id.nunique()} outbursts')

# keep outbursts whose lake was visible before drainage, and lakes with at least one of them
vis = {}
for eid in M.loc[M.label == 1, 'event_id'].unique():
    lid, lo = eid.rsplit('_', 1)[0], pd.Timestamp(eid.rsplit('_', 1)[1])
    d = pd.to_datetime(M.date)
    pre = M[(M.lake_id == lid) & (d < lo) & (d >= lo - pd.Timedelta(days=60))]
    vis[eid] = bool(len(pre)) and float(pre.water_km2.max()) >= MIN_WATER_BEFORE
lake_ok = {lid: any(v for e, v in vis.items() if e.rsplit('_', 1)[0] == lid) for lid in M.lake_id.unique()}
keep = M.lake_id.map(lake_ok).fillna(False).astype(bool) & ~((M.label == 1) & ~M.event_id.map(vis).fillna(False).astype(bool))
M, XO = M[keep].reset_index(drop=True), XO[keep.values]
print(f'kept: {M.lake_id.nunique()} lakes, {len(M)} scenes, {int(M.label.sum())} GLOF from {M[M.label == 1].event_id.nunique()} outbursts')

lakes = np.array(sorted(M.lake_id.unique()))
VAL_LAKES = set(np.random.default_rng(0).choice(lakes, max(1, int(round(VAL_LAKE_FRAC * len(lakes)))), replace=False))
M['val_lake'] = M.lake_id.isin(VAL_LAKES)

# one array: other lakes first, then Shisper
X = np.concatenate([XO, XS]); del XO
y = np.concatenate([M.label.values, S.label.values]).astype('float32')
NO = len(M)
IS_SH = np.arange(len(X)) >= NO
OTHER, OTHER_VAL = ~IS_SH, np.r_[M.val_lake.values, np.zeros(len(S), bool)]
def sh(mask):
    return np.r_[np.zeros(NO, bool), np.asarray(mask, bool)]

## Model, augmentation and training protocol

In [ ]:
def build():
    m = models.Sequential([
        layers.Input((IMG, IMG, 6)),
        layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(2),
        layers.Flatten(), layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')])
    m.compile(tf.keras.optimizers.Adam(1e-3), 'binary_crossentropy', metrics=['accuracy'])
    return m

def fit_norm(idx, rng):
    sub = rng.choice(idx, min(400, len(idx)), replace=False)
    return np.percentile(X[sub], 2, axis=(0, 1, 2)), np.percentile(X[sub], 98, axis=(0, 1, 2))

def norm(x, lo, hi):
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1).astype('float32')

def batches(idx, lo, hi, seed):
    rng = np.random.default_rng(seed)
    pos, neg = idx[y[idx] == 1], idx[y[idx] == 0]
    while True:
        b = np.concatenate([rng.choice(pos, BATCH // 2), rng.choice(neg, BATCH - BATCH // 2)])
        xb = norm(X[b], lo, hi)
        flip_lr, flip_ud = rng.random(len(b)) < .5, rng.random(len(b)) < .5
        xb[flip_lr] = xb[flip_lr][:, :, ::-1]
        xb[flip_ud] = xb[flip_ud][:, ::-1]
        k = rng.integers(0, 4, len(b))
        for r in range(1, 4):
            if (k == r).any():
                xb[k == r] = np.rot90(xb[k == r], r, axes=(1, 2))
        s = rng.uniform(.85, 1.0, len(b))
        y0, x0 = rng.uniform(0, 1 - s), rng.uniform(0, 1 - s)
        boxes = np.stack([y0, x0, y0 + s, x0 + s], 1).astype('float32')
        xb = tf.image.crop_and_resize(xb, boxes, np.arange(len(b), dtype='int32'), [IMG, IMG]).numpy()
        xb = np.clip(xb * rng.uniform(.9, 1.1, (len(b), 1, 1, 1)), 0, 1).astype('float32')
        yield xb, y[b]

def dataset(idx, lo, hi, seed):
    sig = (tf.TensorSpec((None, IMG, IMG, 6), tf.float32), tf.TensorSpec((None,), tf.float32))
    calls = [0]
    def gen():                                   # a fresh random stream each time Keras restarts the iterator
        calls[0] += 1
        return batches(idx, lo, hi, seed * 1000 + calls[0])
    return tf.data.Dataset.from_generator(gen, output_signature=sig).prefetch(2)

def predict(m, idx, lo, hi):
    out = [m.predict(norm(X[idx[i:i + 256]], lo, hi), verbose=0).ravel() for i in range(0, len(idx), 256)]
    return np.concatenate(out) if out else np.array([])

def run_protocol(pool, val, test, seed):
    # stage 1: number of epochs chosen on the validation scenes; stage 2: retrain on the whole pool
    rng = np.random.default_rng(seed)
    tr1, va = np.nonzero(pool & ~val)[0], np.nonzero(val)[0]
    tf.keras.utils.set_random_seed(seed)
    lo, hi = fit_norm(tr1, rng)
    m = build()
    es = tf.keras.callbacks.EarlyStopping('val_loss', patience=PATIENCE, restore_best_weights=True)
    h = m.fit(dataset(tr1, lo, hi, seed), steps_per_epoch=STEPS, epochs=MAX_EPOCHS,
              validation_data=(norm(X[va], lo, hi), y[va]), callbacks=[es], verbose=0)
    ep = int(np.argmin(h.history['val_loss'])) + 1
    tf.keras.backend.clear_session()
    tr2 = np.nonzero(pool)[0]
    tf.keras.utils.set_random_seed(seed)
    lo, hi = fit_norm(tr2, rng)
    m = build()
    m.fit(dataset(tr2, lo, hi, seed + 1000), steps_per_epoch=STEPS, epochs=ep, verbose=0)
    p = predict(m, np.nonzero(test)[0], lo, hi)
    tf.keras.backend.clear_session()
    return p, ep

def run_cached(name, pool, val, test, seed):
    f = f'{OUT}/runs/{name}_seed{seed}.json'
    if os.path.exists(f):
        return np.array(json.load(open(f))['p'])
    t0 = time.time()
    p, ep = run_protocol(pool, val, test, seed)
    json.dump(dict(p=[float(v) for v in p], epochs=ep, minutes=round((time.time() - t0) / 60, 1)), open(f, 'w'))
    print(f'  {name} seed {seed}: {ep} epochs, {(time.time() - t0) / 60:.1f} min', flush=True)
    return p

def wilson(k, n, z=1.96):
    # 95 % Wilson interval for k of n
    if n == 0:
        return ''
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; w = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return f'{max(0.0, c - w):.2f} to {min(1.0, c + w):.2f}'

def summ(lab, prob, t=THRESH):
    lab = np.asarray(lab).astype(int); pred = (np.asarray(prob) > t).astype(int)
    tn, fp, fn, tp = (int(v) for v in confusion_matrix(lab, pred, labels=[0, 1]).ravel())
    return dict(TP=tp, FN=fn, FP=fp, TN=tn, recall_ci95=wilson(tp, tp + fn), false_alarm_rate=round(fp / max(fp + tn, 1), 3),
                false_alarm_ci95=wilson(fp, fp + tn), roc_auc=round(float(roc_auc_score(lab, prob)), 3) if len(set(lab)) == 2 else None)

GLOF_YEARS = sorted(S[S.label == 1].year.unique())
def val_year_for(fold):
    test_years = set(S[S.fold == fold].year)
    return max(yr for yr in GLOF_YEARS if yr not in test_years)

## Test 1: other lakes only

In [ ]:
S['p_other_only'] = np.mean([run_cached('other_only', OTHER, OTHER_VAL, IS_SH, s) for s in SEEDS], axis=0)
R1 = summ(S.label, S.p_other_only)
print('other lakes only, every Shisper scene:', R1)

## Test 2: main split

In [ ]:
trS, vaS, teS = (S.year <= 2021).values, (S.year == 2021).values, (S.year >= 2022).values
S['p_main'] = np.nan
S.loc[teS, 'p_main'] = np.mean([run_cached('main', OTHER | sh(trS), OTHER_VAL | sh(vaS), sh(teS), s) for s in SEEDS], axis=0)
R2 = summ(S.label[teS], S.p_main[teS])
print('main split:', R2)

## Test 3: leave one year out

In [ ]:
S['p_loyo'] = np.nan
for f in sorted(S.fold.unique()):
    te = (S.fold == f).values
    vy = val_year_for(f)
    S.loc[te, 'p_loyo'] = np.mean([run_cached(f'loyo_{f}', OTHER | sh(~te), OTHER_VAL | sh((S.year == vy).values), sh(te), s)
                                   for s in SEEDS], axis=0)
R3 = summ(S.label, S.p_loyo)
print('leave one year out:', R3)

## Results

In [ ]:
ok = (R3['TP'] >= CRITERIA['loyo_min_glof']) and (R3['false_alarm_rate'] < CRITERIA['max_false_alarm_rate']) \
     and (R2['TP'] >= CRITERIA['main_glof'])
res = dict(test1_other_only=R1, test2_main=R2, test3_loyo=R3, seeds=SEEDS, criteria=CRITERIA, criteria_met=bool(ok),
           data=dict(other_lakes=int(M.lake_id.nunique()), other_scenes=int(len(M)), other_glof=int(M.label.sum()),
                     other_outbursts=int(M[M.label == 1].event_id.nunique())))
json.dump(res, open(f'{OUT}/multisite_results.json', 'w'), indent=1)
S.to_csv(f'{OUT}/shisper_predictions_multisite.csv', index=False)
print(json.dumps(res, indent=1))
print(S[S.label == 1][['date', 'p_other_only', 'p_main', 'p_loyo']].round(3).to_string(index=False))

d = pd.to_datetime(S.date)
fig, ax = plt.subplots(figsize=(11, 3.4))
single = f'{SH_DIR}/riskflow_predictions.csv'
if os.path.exists(single):
    O = S[['date']].merge(pd.read_csv(single)[['date', 'p_loyo']], on='date', how='left')
    ax.plot(d, O.p_loyo, 'x', ms=4, c='0.6', label='Shisper only')
ax.scatter(d[S.label == 0], S.p_loyo[S.label == 0], s=12, c='#3a6ea5', label='multi-site, other scene')
ax.scatter(d[S.label == 1], S.p_loyo[S.label == 1], s=46, marker='^', c='#c0392b', label='multi-site, GLOF scene')
for o in ob:
    ax.axvline(o, color='#c0392b', lw=.8, ls='--')
ax.axhline(THRESH, ls=':', c='0.4'); ax.set_ylim(-.03, 1.03); ax.set_ylabel('P(GLOF), year held out')
ax.legend(fontsize=8, loc='upper right', ncol=3)
plt.tight_layout(); plt.savefig(f'{OUT}/fig_multisite_loyo.png', dpi=200); plt.show()